In [1]:
## Initial setup

import os
from dotenv import load_dotenv
import time
import uuid
import json

In [2]:
from portkey_ai import Portkey , createHeaders , PORTKEY_GATEWAY_URL

load_dotenv(dotenv_path="../.env")

PORTKEY_API_KEY = os.getenv("PORTKEY_API_KEY")

portkey = Portkey(api_key=PORTKEY_API_KEY)

### Implementing Virtal keys and slugs

In [3]:
GROQ_SLUG = "gkey"
GROQ_MODEL = f"@{GROQ_SLUG}/openai/gpt-oss-120b"

GROQ_SLUG_2 = "gkey2"
GROQ_MODEL_SMALL = f"@{GROQ_SLUG_2}/openai/gpt-oss-20b"


GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [4]:
print("Setup complete!")
print(f"  Portkey API Key : {'OK' if PORTKEY_API_KEY else 'MISSING'}")
print(f"  Groq slug       : {GROQ_SLUG}")
print(f"  Groq model ref  : {GROQ_MODEL}")
print(f"  Groq slug 2     : {GROQ_SLUG_2}")
print(f"  Small model ref : {GROQ_MODEL_SMALL}")
print(f"\nPortkey Gateway : {PORTKEY_GATEWAY_URL}")

Setup complete!
  Portkey API Key : OK
  Groq slug       : gkey
  Groq model ref  : @gkey/openai/gpt-oss-120b
  Groq slug 2     : gkey2
  Small model ref : @gkey2/openai/gpt-oss-20b

Portkey Gateway : https://api.portkey.ai/v1


## Helper Function

In [5]:
def section(title):
    print(f"\n{'='*62}")
    print(f"  {title}")
    print(f"{'='*62}")

def show(q, answer, ms, label=""):
    bar = chr(9472) * 62
    print(f"\n{bar}")
    print(f"Q: {q}")
    print(f"A: {answer[:260]}{'...' if len(answer) > 260 else ''}")
    note = f" | {label}" if label else ""
    print(f"⏱  {ms:.0f}ms{note}")
    print(bar)


## Simple GROQ call without llm gateway

In [6]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

raw_groq = ChatGroq(
    api_key=GROQ_API_KEY,
    model = "openai/gpt-oss-120b",
    temperature = 0.6,
)

In [7]:
section("Raw Groq Model")

questions = [
    "Write a simple joke on an suncornure parrot",
    "why does coding exist in this world",
]

for q in questions:
    t0 = time.time()
    r = raw_groq.invoke([HumanMessage(content=q)])
    show(q, r.content, (time.time() - t0) * 1000, label="raw Groq - no llm gateway")


  Raw Groq Model

──────────────────────────────────────────────────────────────
Q: Write a simple joke on an suncornure parrot
A: Why did the Sun Conure bring a tiny sunglasses case to the party?

Because it wanted to look **bright** and **shade** its jokes! 🦜😎
⏱  810ms | raw Groq - no llm gateway
──────────────────────────────────────────────────────────────

──────────────────────────────────────────────────────────────
Q: why does coding exist in this world
A: **Short answer:**  
Coding (or programming) exists because humans needed a systematic way to tell machines—especially computers—what to do, and to automate, coordinate, and extend our own problem‑solving abilities.

---

## 1. The Why‑Behind Coding – From a Hu...
⏱  3146ms | raw Groq - no llm gateway
──────────────────────────────────────────────────────────────


##  EXPERIMENT 1 — Route Through the Gateway

In [8]:
section("EXP! - Basic Portkey Gateway")

questions = [
    "Tell me difference between gen AI vs Agentic AI",
    "Tell me how black coffee differs from noremal coffee and what are its benefits"
]

for q in questions:
    t0 = time.time()
    r = portkey.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role":"user","content":q}]
    )
    show(q, r.choices[0].message.content, (time.time()-t0)*1000,
         label="routed via Portkey gateway")
    
print("\n Check portkey.ai → Logs to see both requests fully logged!")
print("   Token count, cost, latency — all tracked. Zero extra code.")


  EXP! - Basic Portkey Gateway

──────────────────────────────────────────────────────────────
Q: Tell me difference between gen AI vs Agentic AI
A: **Generative AI vs. Agentic AI – a quick‑look comparison**

| Aspect | **Generative AI** | **Agentic AI** |
|--------|-------------------|----------------|
| **Core purpose** | Produce new artefacts (text, images, audio, code, etc.) that resemble the data it w...
⏱  3712ms | routed via Portkey gateway
──────────────────────────────────────────────────────────────

──────────────────────────────────────────────────────────────
Q: Tell me how black coffee differs from noremal coffee and what are its benefits
A: **Black coffee vs. “regular” (i.e., additive‑laden) coffee**

| Aspect | Black Coffee | Regular Coffee (with milk, sugar, cream, flavored syrups, etc.) |
|--------|--------------|---------------------------------------------------------------|
| **Definition**...
⏱  3807ms | routed via Portkey gateway
────────────────────────────────

##  EXPERIMENT 2 — Metadata & Observability

In [9]:
section("EXP 2 — Metadata & Observability")

# new unique id

session = str(uuid.uuid4())[:8]

scenarios = [
    ("alice", "enterprise-rag",   "What is Kubernetes RBAC?"),
    ("bob",   "docs-chatbot",     "How does BGP path selection work?"),
    ("carol", "support-bot",      "What is SRIOV virtualization?"),
    ("alice", "enterprise-rag",   "Explain Kubernetes NetworkPolicy"),   # same user, diff Q
]




  EXP 2 — Metadata & Observability


In [10]:
for user, feature, q in scenarios:
    t0 = time.time()
    r = portkey.with_options(
        metadata={
            "_user":       user,          # powers per-user analytics in dashboard
            "session_id":  session,
            "feature":     feature,
            "environment": "notebook"
        }
    ).chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": q}]
    )
    ms = (time.time() - t0) * 1000
    print(f"\n👤 {user:8s} | 🔧 {feature:18s} | {ms:.0f}ms")
    print(f"  Q: {q}")
    print(f"  A: {r.choices[0].message.content[:120]}...")


👤 alice    | 🔧 enterprise-rag     | 4631ms
  Q: What is Kubernetes RBAC?
  A: ## Kubernetes RBAC – Role‑Based Access Control

Kubernetes RBAC is the **authorization mechanism** that lets you define ...

👤 bob      | 🔧 docs-chatbot       | 5756ms
  Q: How does BGP path selection work?
  A: **Border Gateway Protocol (BGP)** is the “glue” that holds the Internet together.  
When a BGP speaker (router) receives...

👤 carol    | 🔧 support-bot        | 6166ms
  Q: What is SRIOV virtualization?
  A: **SR‑IOV (Single‑Root I/O Virtualization)** is a hardware‑assisted technology that allows a single physical PCI‑Express ...


RateLimitError: Error code: 429 - {'error': {'message': 'groq error: Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kn3qaamzerebgkymt4xpjpp5` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7994, Requested 1391. Please try again in 10.3875s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}, 'provider': 'groq'}

## Experiment 3 Automatic Retries

In [14]:

portkey_retry = Portkey(api_key=PORTKEY_API_KEY, config="pc-portke-b83816")

section("EXP 3 — Automatic Retries")
print("Config: 3 retry attempts on [429, 500, 502, 503, 504, 400]")
print("Retries fire automatically on failure — transparent to your code\n")


try:
    t0 = time.time()
    r = portkey_retry.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": "What is a AI?"}]
    )
    ms = (time.time() - t0) * 1000
    print(f"✅ Succeeded in {ms:.0f}ms")
    print(f"   {r.choices[0].message.content[:300]}")
    print("\nRetry sequence if Groq had failed:")
    print("  Attempt 1 → 429 → wait 1s → Attempt 2 → 429 → wait 2s → Attempt 3")
    print("  Your code only sees the final success or the last failure")
except Exception as e:
    print(f"❌ All attempts failed: {e}")


  EXP 3 — Automatic Retries
Config: 3 retry attempts on [429, 500, 502, 503, 504, 400]
Retries fire automatically on failure — transparent to your code

✅ Succeeded in 4772ms
   **Artificial Intelligence (AI)** is a broad field of computer science focused on creating systems that can perform tasks that normally require human intelligence. These tasks include learning, reasoning, problem‑solving, perception, language understanding, and even creativity. 

Below is a concise o

Retry sequence if Groq had failed:
  Attempt 1 → 429 → wait 1s → Attempt 2 → 429 → wait 2s → Attempt 3
  Your code only sees the final success or the last failure


## Experiment 4 - Request Timeout

In [15]:
portkey_timeout = Portkey(api_key=PORTKEY_API_KEY, config="pc-timeou-6c2161")

section("EXP 4 — Request Timeouts")
print("Timeout: 10,000ms (10 seconds). Portkey returns HTTP 408 if exceeded.\n")

try:
    t0 = time.time()
    r = portkey_timeout.chat.completions.create(
        model=GROQ_MODEL_SMALL,
        messages=[{"role": "user", "content": "Explain Kubernetes networking in 2 sentences."}]
    )
    ms = (time.time() - t0) * 1000
    print(f"✅ Response in {ms:.0f}ms (within 10s timeout)")
    print(f"   {r.choices[0].message.content}")
except Exception as e:
    print(f"⏱  Timed out: {e}")
    print("   Portkey issued a 408. Pair with fallback to auto-switch providers on timeout.")

print("\n--- Combining timeout + retry (production pattern) ---")
combined = {
    "request_timeout": 10000,
    "retry": {"attempts": 2, "on_status_codes": [408, 429, 503]}
}
print(json.dumps(combined, indent=2))


  EXP 4 — Request Timeouts
Timeout: 10,000ms (10 seconds). Portkey returns HTTP 408 if exceeded.

✅ Response in 2765ms (within 10s timeout)
   Kubernetes networking is a flat, cluster‑wide network that assigns every pod a unique IP address and exposes services via stable, cluster‑wide DNS names, allowing pods to communicate with one another regardless of node placement. Service objects then load‑balance traffic to the appropriate pods, and NetworkPolicies can restrict that traffic, ensuring secure, scalable inter‑pod communication.

--- Combining timeout + retry (production pattern) ---
{
  "request_timeout": 10000,
  "retry": {
    "attempts": 2,
    "on_status_codes": [
      408,
      429,
      503
    ]
  }
}


## EXPERIMENT 5 — Fallbacks

In [16]:
portkey_fallback = Portkey(api_key=PORTKEY_API_KEY, config="pc-fallba-ea70df")

section("EXP 5 — Fallback Routing")
print(f"Primary  : {GROQ_MODEL}")
print(f"Fallback : {GROQ_MODEL_SMALL}\n")

fallback_questions = [
    "What is Intel Technology?",
    "Explain me jarvis in detail.",
]

for q in fallback_questions:
    try:
        t0 = time.time()
        r = portkey_fallback.chat.completions.create(
            messages=[{"role": "user", "content": q}]
        )
        ms = (time.time() - t0) * 1000
        show(q, r.choices[0].message.content, ms, label="primary served")
    except Exception as e:
        print(f"❌ {e}")
        print("   → Check that both GROQ_SLUG and GROQ_SLUG_2 are set correctly in c04")


  EXP 5 — Fallback Routing
Primary  : @gkey/openai/gpt-oss-120b
Fallback : @gkey2/openai/gpt-oss-20b


──────────────────────────────────────────────────────────────
Q: What is Intel Technology?
A: **Intel Technology** is a broad term that refers to the family of semiconductor devices, software, and systems developed and sold by **Intel Corporation**—a global leader in computer and data‑processing technologies.  Below is a concise overview of what “Intel...
⏱  3096ms | primary served
──────────────────────────────────────────────────────────────

──────────────────────────────────────────────────────────────
Q: Explain me jarvis in detail.
A: Sure! Could you let me know which *Jarvis* you’re referring to?  
- **Marvel’s JARVIS** – the AI assistant from Iron Man (Just A Rather Very Intelligent System).  
- **Jarvis software/library** – an open‑source tool or framework (e.g., JARVIS for Java, a Java-...
⏱  757ms | primary served
──────────────────────────────────────────────────────────

## Forcing Model to be failed

In [17]:
# ── Part 2: FORCED fallback — bad primary key triggers real failover ──
print("\n\n--- FORCED FALLBACK DEMO ---")
print("Primary target uses a deliberately invalid Groq API key.")
print("Groq returns 401 → Portkey detects non-2xx → fallback fires automatically.\n")

portkey_forced = Portkey(api_key=PORTKEY_API_KEY, config="pc-forced-cd1c51")

try:
    t0 = time.time()
    r = portkey_forced.chat.completions.create(
        messages=[{"role": "user", "content": "What is ge ai?"}]
    )
    ms = (time.time() - t0) * 1000
    print(f"✅ Got a response in {ms:.0f}ms despite the bad primary key!")
    print(f"   {r.choices[0].message.content[:250]}")
    print("\n→ Check Portkey Logs: attempt 1 shows FAILED (401), attempt 2 shows SUCCEEDED")
    print("   The fallback fired automatically — app code never saw the error.")
except Exception as e:
    print(f"❌ Both targets failed: {e}")



--- FORCED FALLBACK DEMO ---
Primary target uses a deliberately invalid Groq API key.
Groq returns 401 → Portkey detects non-2xx → fallback fires automatically.

✅ Got a response in 2514ms despite the bad primary key!
   "That" is a typo in question; based on context, they probably meant "What is **AI**?".

**AI (Artificial Intelligence)** is a branch of computer science that focuses on creating systems capable of performing tasks that typically require human intelli

→ Check Portkey Logs: attempt 1 shows FAILED (401), attempt 2 shows SUCCEEDED
   The fallback fired automatically — app code never saw the error.
